# 구조 정리
PDF 로드
→ 영역별 텍스트 추출
→ 조항 split
→ metadata 생성
→ 결과 확인
→ JSON 저장

1. PDF 로딩 (PyMuPDFLoader)
2. 조항 단위 split
3. metadata 추가
4. embedding 생성
5. vector store 저장 (FAISS/Chroma)
6. retrieval + QA

In [ ]:
!pip install -q langchain langchain-community pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import glob
import re
import json

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

In [ ]:
pdf_files = glob.glob('/content/*.pdf')

print("불러온 PDF 목록")
for pdf in pdf_files:
    print(pdf)

불러온 PDF 목록
/content/삼성화재_실손의료보험_약관.pdf


# 1. metadata 생성 및 저장

삼성화재 실손의료비보험 약관 pdf로만 일단 진행함

- 보통약관(p.24)
- 특별약관(p.49)
- 별표 및 제도성 특별약관(p.75)
- 약관에서 인용된 법, 규정(p.113)



In [ ]:
#  PDF 로드

pdf_path = "/content/삼성화재_실손의료보험_약관.pdf"

loader = PyMuPDFLoader(pdf_path)
pages = loader.load()

print(f"총 페이지 수: {len(pages)}")

총 페이지 수: 157


In [ ]:
# 문서 구간 정의

# 실제 페이지 기준
#
# 보통약관(p.24)
# 특별약관(p.49)
# 별표 및 제도성 특별약관(p.75)
# 법·규정(p.113)
#
# PyMuPDFLoader는 index가 0부터 시작


sections = {
    "보통약관": (23, 48),
    "특별약관": (48, 74),
    "별표및제도성특약": (74, 112),
    "법규": (112, 154)
}


# 최종 Document 저장 리스트
docs = []

# 문서 유형별 처리
for document_type, (start_page, end_page) in sections.items():

    print(f"\n[{document_type}] 처리 중")

    # 페이지 범위 추출
    target_pages = pages[start_page:end_page]

    # 텍스트 합치기
    full_text = "\n".join(
        [p.page_content for p in target_pages]
    )

    # 관 + 조항 기준 split
    pattern = r"(제\d+관\s+[^\n]+|제\d+조\s*\([^)]+\))"
    splits = re.split(pattern, full_text)
    current_section = "Unknown"

    # 조항 단위 처리
    for i in range(1, len(splits), 2):

        title = splits[i]
        content = splits[i + 1] if i + 1 < len(splits) else ""

        # 관(section) 저장
        if "관" in title:
            current_section = title.strip()
            continue

        # 조항 chunk 생성
        chunk = title + "\n" + content

        # 너무 짧은 chunk 제거
        if len(chunk.strip()) < 30:
            continue

        # 조항 번호
        article_match = re.search(
            r"(제\d+조)",
            chunk
        )

        article = (
            article_match.group(1)
            if article_match
            else "Unknown"
        )

        # 조항 제목
        title_match = re.search(
            r"제\d+조\s*\(([^)]+)\)",
            chunk
        )

        article_title = (
            title_match.group(1)
            if title_match
            else "Unknown"
        )

        # metadata
        metadata = {
            "company": "삼성화재",
            "document_type": document_type,
            "section": current_section,
            "article": article,
            "title": article_title
        }

        # LangChain Document 생성
        doc = Document(
            page_content=chunk,
            metadata=metadata
        )

        docs.append(doc)


[보통약관] 처리 중

[특별약관] 처리 중

[별표및제도성특약] 처리 중

[법규] 처리 중


In [ ]:
print(f"\n총 생성된 Document 수: {len(docs)}")


총 생성된 Document 수: 315


In [ ]:
for i in range(5):

    print("\n" + "=" * 80)

    print(docs[i].metadata)

    print("\n[본문 일부]\n")

    print(docs[i].page_content[:500])


{'company': '삼성화재', 'document_type': '보통약관', 'section': '제1관 일반사항 및 용어의 정의', 'article': '제1조', 'title': '보장종목'}

[본문 일부]

제1조 (보장종목)
 
① 회사가 판매하는 기본형 실손의료보험상품은 다음과 같이 상해급여형, 질병급여형의 
2개 보장종목으로 구성되어 있습니다. 
보장종목 
보상하는 내용 
상해급여 
피보험자가 상해로 인하여 의료기관에 입원 또는 통원하여 급여주) 치료를 받거
나 급여 처방조제를 받은 경우에 보상 
질병급여 
피보험자가 질병으로 의료기관에 입원 또는 통원하여 급여 치료를 받거나 급여 
처방조제를 받은 경우에 보상 
주)「국민건강보험법」에서 정한 요양급여 또는 「의료급여법」에서 정한 의료급여 
② 회사는 이 약관의 명칭에 ‘급여 실손의료비’라는 문구를 포함하여 사용합니다. 
 


{'company': '삼성화재', 'document_type': '보통약관', 'section': '제1관 일반사항 및 용어의 정의', 'article': '제2조', 'title': '용어의 정의'}

[본문 일부]

제2조 (용어의 정의)
 
이 약관에서 사용하는 용어의 뜻은 [붙임1]과 같습니다.  
 


{'company': '삼성화재', 'document_type': '보통약관', 'section': '제2관 회사가 보상하는 사항', 'article': '제3조', 'title': '보장종목별 보상내용'}

[본문 일부]

제3조 (보장종목별 보상내용)
 
회사가 이 계약의 보험기간 중 보장종목별로 각각 보상하거나 공제하는 내용은 다음과 
같습니다. 
 
(1) 상해급여 
① 회사는 피보험자가 상해로 인하여 의료기관에 입원 또는 통원(외래 및 처방조제)하여 
치료를 받은 경우에는 급여의료비를 

{'company': '삼성화재', 'document_type': '보통약관', 'section': '제2관 회사가 보상하는 사항', 'article': '제6조'

### JSON 저장

In [ ]:
save_data = []

for doc in docs:

    save_data.append({

        "page_content": doc.page_content,

        "metadata": doc.metadata
    })


with open(
    "samsung_insurance_clause_dataset.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        save_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nJSON 저장 완료")


JSON 저장 완료


# 2. 벡터 DB 저장

Chroma 벡터스토어로 보험 조항들을 저장해두는 검색용 DB 생성

In [ ]:
!pip install chromadb
!pip install sentence-transformers
!pip install langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


### 1) 임베딩 모델 준비
- 한국어 검색 성능 좋은 HuggingFace 사용

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask"
)

/tmp/ipykernel_15961/2126577141.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 2) Chroma 저장

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print("Chroma 벡터스토어 저장 완료")

Chroma 벡터스토어 저장 완료


### 3) 검색 테스트

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

query = "자기부담금"

results = retriever.invoke(query)

for r in results:
    print("=" * 80)
    print(r.metadata)
    print(r.page_content[:500])

{'company': '삼성화재', 'section': '제9조(의료급여기관)', 'article': '제10조', 'title': '급여비용의 부담', 'document_type': '법규'}
제10조(급여비용의 부담)
 
급여비용은 대통령령으로 정하는 바에 따라 그 전부 또는 일부를 제25조에 따른 의료급
여기금에서 부담하되, 의료급여기금에서 일부를 부담하는 경우 그 나머지 비용은 본인이 
부담한다. 

{'article': '제13조', 'title': '급여비용의 부담', 'section': '제3조(의료기관)', 'company': '삼성화재', 'document_type': '보통약관'}
제13조(급여비용의 부담)
에서 정하는 금액을 넘는 경우에 그 초과한 
금액을 의료급여기금 등에서 부담하는 제도를 말하며, 의료급여 관련 법령
용어 
정의 
본인부담금 상한제 
의 변경에 따라 환급기준이 변경될 경우에는 회사는 변경된 기준에 따름 
다수보험 
실손의료보험계약(우체국보험, 각종 공제, 상해∙질병∙간병보험 등 제3보
험, 개인연금∙퇴직보험 등 의료비를 실손으로 보상하는 보험∙공제계약을 
포함)이 동시에 또는 순차적으로 2개 이상 체결되었고, 그 계약이 동일
한 보험사고에 대하여 각 계약별 보장책임액이 있는 여러 개의 실손의료
보험계약을 말함 
보장대상의료비 
실제 부담액 – 보장제외금액* 
* 
{'section': '제20조(약관 교부 및 설명 의무 등)', 'article': '제6조', 'document_type': '보통약관', 'title': '보험가\n입금액 한도 등', 'company': '삼성화재'}
제6조(보험가
입금액 한도 등)
 제3항의 ‘본인부담금 상한제’ 및 ‘본인부담금 보상제’에 대한 사항을 
구체적으로 설명하여 드립니다. 
<용어풀이> 
[「국민건강보험법」에 따른 본인부담금 상한제] 
요양급여비용 중 본인이 부담한 비용의 연간 총액이 일정 상한액(국민건강보험 지역가입자의 세대
별 보험료 부담수준 또는 직장가입자의 개인별 보험료